# Logical Consistency Validation

This notebook checks whether the current validation outputs are internally coherent and highlights
whether the merged pipeline is thesis-ready or still needs reruns.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from walaris.notebook_support import (
    ensure_columns,
    find_project_root,
    load_sweep_metrics,
    summarize_trends,
    validate_basic_bounds,
    validate_decomposition,
    validation_dir,
)

plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")

project_root = find_project_root()
results_root = validation_dir(project_root)
print(f"Project root: {project_root}")
print(f"Validation root: {results_root}")


In [ ]:
dataset_loaded = load_sweep_metrics("dataset_size", project_root=project_root)
noise_loaded = load_sweep_metrics("label_noise", project_root=project_root)

for warning in [*dataset_loaded.warnings, *noise_loaded.warnings]:
    print(f"WARNING: {warning}")

df_dataset = dataset_loaded.df.copy()
df_noise = noise_loaded.df.copy()
df_all = pd.concat([df_dataset, df_noise], ignore_index=True)

print(f"Dataset rows: {len(df_dataset)}")
print(f"Label-noise rows: {len(df_noise)}")
print(f"Combined rows: {len(df_all)}")

display(
    ensure_columns(
        df_all,
        [
            "architecture",
            "sweep_type",
            "dataset_size",
            "noise_rate",
            "noise_percent",
            "accuracy",
            "mean_epistemic_uncertainty",
            "mean_aleatoric_uncertainty",
            "mean_total_uncertainty",
        ],
    ).head(20)
)


In [ ]:
df_decomposition = validate_decomposition(df_all)
df_bounds = validate_basic_bounds(df_all)

display(df_decomposition.head())
display(df_bounds)

if not df_decomposition.empty:
    print(
        f"Decomposition pass rate: "
        f"{df_decomposition['passed'].mean() * 100:.1f}%"
    )


In [ ]:
dataset_trends = summarize_trends(df_dataset, "dataset_size")
noise_trends = summarize_trends(df_noise, "label_noise")
trend_summary = pd.concat([dataset_trends, noise_trends], ignore_index=True)
display(trend_summary)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if not df_decomposition.empty:
    axes[0].scatter(df_decomposition["total_expected"], df_decomposition["total_measured"], alpha=0.7)
    max_val = max(df_decomposition["total_expected"].max(), df_decomposition["total_measured"].max())
    axes[0].plot([0, max_val], [0, max_val], "k--", alpha=0.6)
    axes[0].set_title("Measured vs Expected Total Uncertainty")
    axes[0].set_xlabel("Expected")
    axes[0].set_ylabel("Measured")

pass_counts = trend_summary["passed"].value_counts(dropna=False).to_dict() if not trend_summary.empty else {}
axes[1].bar(
    [str(key) for key in pass_counts.keys()],
    list(pass_counts.values()),
    color=["#2ca02c", "#d62728", "#7f7f7f"][: len(pass_counts)],
)
axes[1].set_title("Trend Check Outcomes")
axes[1].set_xlabel("passed")
axes[1].set_ylabel("count")

plt.tight_layout()
plt.show()


In [ ]:
report_lines = []
report_lines.append("LOGICAL CONSISTENCY REPORT")
report_lines.append("=" * 80)
report_lines.append(f"Dataset rows: {len(df_dataset)}")
report_lines.append(f"Label-noise rows: {len(df_noise)}")
report_lines.append("")

if noise_loaded.warnings:
    report_lines.append("WARNINGS:")
    report_lines.extend(f"- {warning}" for warning in noise_loaded.warnings)
    report_lines.append("")

if not df_decomposition.empty:
    report_lines.append(
        f"Decomposition pass rate: {df_decomposition['passed'].mean() * 100:.1f}%"
    )

if not df_bounds.empty:
    failed_bounds = df_bounds[~df_bounds["passed"]]
    report_lines.append(f"Bounds failures: {len(failed_bounds)}")

if not trend_summary.empty:
    passed_rows = trend_summary["passed"] == True
    failed_rows = trend_summary["passed"] == False
    report_lines.append(f"Trend checks passed: {int(passed_rows.sum())}")
    report_lines.append(f"Trend checks failed: {int(failed_rows.sum())}")

report_path = results_root / "consistency_checks" / "validation_report.txt"
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text("\n".join(report_lines))
print(report_path.read_text())
print(f"Saved report to: {report_path}")
